In [ ]:
import pandas as pd
import numpy as np

In [25]:
data = pd.read_csv("reddit_text_10000_labeled.csv")
data.head()

,title,description,comments,comment_toxicity,num_toxic_comments
0,So what are auctioned houses going for compare...,Thinking about the Waikato. We are in the mark...,"[""You can generally look up auction results an...","[0.0006122507620602846, 0.000645090127363801, ...",0
1,Hows the Architecture Industry,I'm Canadian but grew up and work in Hong Kong...,"[""I’m an engineer in AEC industry, there is wo...","[0.0008482924313284457, 0.0006549436948262155,...",0
2,Small House Storage Ideas/Solutions?,"By pure luck, my husband and I were able to pu...","[""Under-bed storage boxes are game changers an...","[0.024000633507966995, 0.0007206632290035486, ...",0
3,Worked at a StartUp it Sucked,Warning: ask lots of questions before you join...,"[""Sorry for your experience..."", ""Zero reading...","[0.0006490582600235939, 0.0057919640094041824,...",0
4,"[HIRING] Backend, Infra & Agentic AI Engineers...",I work with a curated talent network that matc...,"[""do you hire remote ? I've over 3 YOE with Te...","[0.0028175250627100468, 0.0006366739980876446,...",0


In [26]:
import json

# Post-level cutoff, distinct from the comment model's 0.5 score threshold.
# 4% gives the closest available balance on the completed 1,000-post sample.
# 558 posts have exactly 0% toxicity, so no percentage cutoff can give 50/50.
TOXIC_PERCENT_THRESHOLD = 10
if not 0 < TOXIC_PERCENT_THRESHOLD <= 100:
    raise ValueError("TOXIC_PERCENT_THRESHOLD must be in (0, 100]")

def count_comments(value):
    comments = json.loads(value)
    if not isinstance(comments, list) or not all(isinstance(text, str) for text in comments):
        raise ValueError("Expected comments to be a JSON array of strings")
    return len(comments)

data["num_comments"] = data["comments"].apply(count_comments)
toxic_counts = pd.to_numeric(data["num_toxic_comments"], errors="raise")
if (data["num_comments"] == 0).any():
    raise ValueError("Cannot calculate toxic percentage for posts with no comments")
if (toxic_counts.isna().any() or not np.isfinite(toxic_counts).all()
        or (toxic_counts < 0).any() or (toxic_counts > data["num_comments"]).any()
        or (toxic_counts % 1 != 0).any()):
    raise ValueError("Toxic comment counts must be integers between 0 and num_comments")

data["toxic_comment_percentage"] = 100 * toxic_counts / data["num_comments"]
# Apply the cutoff before any rounding for display.
data["toxic_index"] = (data["toxic_comment_percentage"] >= TOXIC_PERCENT_THRESHOLD).astype(int)
TARGET_NAMES = [f"0: below {TOXIC_PERCENT_THRESHOLD:g}% toxic",
                f"1: at least {TOXIC_PERCENT_THRESHOLD:g}% toxic"]

toxic_index_counts = data["toxic_index"].value_counts().reindex([0, 1], fill_value=0)
toxic_index_counts.index.name = "toxic_index"
toxic_index_counts.name = "number_of_posts"
print(f"Positive target: at least {TOXIC_PERCENT_THRESHOLD:g}% of comments are toxic")
print(toxic_index_counts.to_string())
print(data[["num_comments", "num_toxic_comments", "toxic_comment_percentage", "toxic_index"]].head())


Positive target: at least 10% of comments are toxic
toxic_index
0    7776
1    2224
   num_comments  num_toxic_comments  toxic_comment_percentage  toxic_index
0            10                   0                       0.0            0
1            10                   0                       0.0            0
2            16                   0                       0.0            0
3            12                   0                       0.0            0
4             6                   0                       0.0            0


## Predict whether a post attracts toxic comments with a linear SVM

Use **title + description** as input and `toxic_index` as the target (1 = the toxic-comment percentage meets or exceeds `TOXIC_PERCENT_THRESHOLD`, default 10%). Comments and toxicity scores are excluded from the input because they define the target. These are model-generated labels, not human annotations.

Hold out 20% of posts for evaluation. Learn the TF-IDF vocabulary only from training posts, then train a linear support vector machine. Exact duplicate input texts are kept in the same split to prevent leakage.

The target is `100 × num_toxic_comments / num_comments`. For example, 1 toxic comment out of 25 is 4% (positive), while 1 out of 100 is 1% (negative). This normalizes by comment volume, but percentages from a few comments are still less stable. The 4% cutoff gives 442 positive and 558 negative posts on the completed 1,000-post sample, the closest possible balance because 558 posts have zero toxic comments. On this sample, it yields the same labels as the original any-toxic-comment rule; a larger sample may differ. Keep the threshold fixed before evaluating new data. Count and percentage columns are target metadata, not model inputs. Previous model scores used a different target and must be recomputed.


In [27]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score

X = data["title"].fillna("").astype(str) + " " + data["description"].fillna("").astype(str)
y = data["toxic_index"]

# Split unique normalized texts, so duplicate posts cannot cross the split.
groups = X.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
group_labels = pd.DataFrame({"group": groups, "label": y}).groupby("group", sort=False)["label"].max()
train_groups, test_groups = train_test_split(
    group_labels.index.to_numpy(), test_size=0.2, random_state=42, stratify=group_labels
)
train_mask = groups.isin(train_groups)
test_mask = groups.isin(test_groups)
X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]
assert set(groups.loc[train_mask]).isdisjoint(groups.loc[test_mask])
print(f"Training posts: {len(X_train)}; test posts: {len(X_test)}")
print(pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()}).sort_index())


Training posts: 7999; test posts: 2001
             train  test
toxic_index             
0             6219  1557
1             1780   444


## Tokenization and TF-IDF

The vectorizer lowercases text and splits it into word tokens of at least two characters, ignoring punctuation. It uses single words (unigrams) and adjacent word pairs (bigrams). TF-IDF weights terms by their frequency within a post and their rarity across training posts. The SVM receives these numeric features, not raw text.


In [28]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True, ngram_range=(1, 2), min_df=2,
        max_features=30000, sublinear_tf=True
    )),
    ("svm", LinearSVC(C=1.0, class_weight="balanced", random_state=42, max_iter=10000)),
])

vectorizer = svm_pipeline.named_steps["tfidf"]
example_text = X_train.iloc[0]
tokens = vectorizer.build_tokenizer()(vectorizer.build_preprocessor()(example_text))
print("Text preview:", example_text[:400])
print("First 30 word tokens:", tokens[:30])
print("First 40 unigram/bigram features:", vectorizer.build_analyzer()(example_text)[:40])


Text preview: So what are auctioned houses going for compared to their ratable values right now? Thinking about the Waikato. We are in the market for a home and I keep seeing auctions pop up. So are they worth going for? Just wondering your general opinion and what they're going for on average compared to their ratable values.
First 30 word tokens: ['so', 'what', 'are', 'auctioned', 'houses', 'going', 'for', 'compared', 'to', 'their', 'ratable', 'values', 'right', 'now', 'thinking', 'about', 'the', 'waikato', 'we', 'are', 'in', 'the', 'market', 'for', 'home', 'and', 'keep', 'seeing', 'auctions', 'pop']
First 40 unigram/bigram features: ['so', 'what', 'are', 'auctioned', 'houses', 'going', 'for', 'compared', 'to', 'their', 'ratable', 'values', 'right', 'now', 'thinking', 'about', 'the', 'waikato', 'we', 'are', 'in', 'the', 'market', 'for', 'home', 'and', 'keep', 'seeing', 'auctions', 'pop', 'up', 'so', 'are', 'they', 'worth', 'going', 'for', 'just', 'wondering', 'your']


In [29]:
# fit learns both TF-IDF weights and the SVM from training data only.
svm_pipeline.fit(X_train, y_train)

example_vector = vectorizer.transform([example_text])
feature_names = vectorizer.get_feature_names_out()
example_features = pd.DataFrame({
    "feature": feature_names[example_vector.indices],
    "tfidf_weight": example_vector.data,
}).sort_values("tfidf_weight", ascending=False)
print(f"Training feature matrix shape: ({len(X_train)}, {len(feature_names)})")
print("Highest-weight features for the example post:")
print(example_features.head(15).to_string(index=False))


Training feature matrix shape: (7999, 30000)
Highest-weight features for the example post:
    feature  tfidf_weight
  going for      0.331175
     values      0.238650
   to their      0.214507
compared to      0.202573
   compared      0.197463
     so are      0.182222
opinion and      0.177898
     pop up      0.174243
keep seeing      0.161462
 market for      0.160494
   for home      0.160494
 on average      0.156171
   for just      0.155393
      up so      0.155393
        pop      0.149949


## Tune the SVM using training data only

Compare 32 configurations with three-fold stratified group cross-validation. Duplicate normalized texts stay together in every fold. Each fold learns its own vocabulary and TF-IDF weights. Search word unigrams versus unigrams + bigrams, minimum document frequency, SVM regularization (`C`), and class weighting. Select by mean toxic-class F1; also report balanced accuracy. The original SVM settings are included in the search.

One worker limits competition with the running toxicity-labeling job. Temporary caching reuses TF-IDF transformations across SVM settings and is removed after fitting. The held-out test set is used only for the final comparison; tuning does not guarantee a higher test score.


In [30]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import make_scorer
from joblib import Memory
from tempfile import TemporaryDirectory

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
train_group_ids = groups.loc[X_train.index].to_numpy()
cv_splits = list(cv.split(X_train, y_train, groups=train_group_ids))
for fit_indices, validation_indices in cv_splits:
    assert set(train_group_ids[fit_indices]).isdisjoint(train_group_ids[validation_indices])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "svm__C": [0.1, 1.0, 10.0, 0.01],
    "svm__class_weight": [None, "balanced"],
}
# 2 x 2 x 4 x 2 = 32 candidates, each evaluated on 3 training folds.
with TemporaryDirectory(prefix="reddit_svm_cache_") as cache_dir:
    candidate = clone(svm_pipeline).set_params(memory=Memory(cache_dir, verbose=0))
    search = GridSearchCV(
        candidate, param_grid=param_grid, cv=cv_splits,
        scoring={"toxic_f1": make_scorer(f1_score, zero_division=0),
                 "balanced_accuracy": "balanced_accuracy"},
        refit="toxic_f1", n_jobs=1, verbose=1, error_score="raise",
    )
    search.fit(X_train, y_train)
    tuned_svm = search.best_estimator_.set_params(memory=None)

print("Selected parameters:", search.best_params_)
print(f"Mean cross-validation toxic F1: {search.best_score_:.3f}")
cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_toxic_f1")
print(cv_results[["mean_test_toxic_f1", "std_test_toxic_f1",
                  "mean_test_balanced_accuracy", "params"]].head(5).to_string(index=False))


Fitting 3 folds for each of 32 candidates, totalling 96 fits
Selected parameters: {'svm__C': 0.01, 'svm__class_weight': 'balanced', 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
Mean cross-validation toxic F1: 0.520
 mean_test_toxic_f1  std_test_toxic_f1  mean_test_balanced_accuracy                                                                                              params
           0.520334           0.013889                     0.706279 {'svm__C': 0.01, 'svm__class_weight': 'balanced', 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
           0.518908           0.008780                     0.700369  {'svm__C': 0.1, 'svm__class_weight': 'balanced', 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
           0.518598           0.014376                     0.701844 {'svm__C': 0.01, 'svm__class_weight': 'balanced', 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
           0.518557           0.013032                     0.705032 {'svm__C': 0.01, 'svm__class_weight': 'b

## Evaluate on held-out posts

Compare against a baseline that always predicts the most common training class. For class 1, precision measures how often a predicted toxic-comment post is correct; recall measures how many actual toxic-comment posts are found. The confusion matrix has actual classes in rows and predicted classes in columns. This measures agreement with the toxicity model's labels, not verified toxicity or causation.


In [24]:
original_pred = svm_pipeline.predict(X_test)
y_pred = tuned_svm.predict(X_test)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(np.zeros((len(y_train), 1)), y_train)
baseline_pred = baseline.predict(np.zeros((len(y_test), 1)))

metrics = pd.DataFrame([
    {"model": name, "accuracy": accuracy_score(y_test, predictions),
     "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
     "toxic_f1": f1_score(y_test, predictions, zero_division=0)}
    for name, predictions in [("Majority baseline", baseline_pred), ("Original SVM", original_pred), ("Tuned SVM", y_pred)]
])
print(metrics.to_string(index=False))
print("\nSVM classification report:")
print(classification_report(y_test, y_pred, labels=[0, 1],
    target_names=TARGET_NAMES, zero_division=0))
confusion = pd.DataFrame(confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=["actual 0", "actual 1"], columns=["predicted 0", "predicted 1"])
print("Confusion matrix:")
print(confusion)


            model  accuracy  balanced_accuracy  toxic_f1
Majority baseline  0.771144           0.500000  0.000000
     Original SVM  0.741294           0.541795  0.235294
        Tuned SVM  0.701493           0.607714  0.400000

SVM classification report:
                       precision    recall  f1-score   support

   0: below 10% toxic       0.82      0.78      0.80       155
1: at least 10% toxic       0.37      0.43      0.40        46

             accuracy                           0.70       201
            macro avg       0.60      0.61      0.60       201
         weighted avg       0.72      0.70      0.71       201

Confusion matrix:
          predicted 0  predicted 1
actual 0          121           34
actual 1           26           20


In [23]:
# Inspect held-out predictions. Decision scores are margins, not probabilities.
test_predictions = data.loc[X_test.index, ["title", "toxic_index"]].copy()
test_predictions["predicted_toxic_index"] = y_pred
test_predictions["svm_decision_score"] = tuned_svm.decision_function(X_test)
test_predictions.head(10)


,title,toxic_index,predicted_toxic_index,svm_decision_score
1,Financial Advice,0,0,-0.070594
7,CMV: I don’t see anything wrong with “the male...,0,0,-0.008577
16,AITA for taking my MOH out of the wedding party?,1,1,0.052869
18,Best place to buy engagement ring,0,0,-0.065447
19,Has anyone seen any now hiring signs at local ...,0,0,-0.039792
21,The US needs China to rebuild their arsenal,1,1,0.085317
26,One gut bacterium tracks with suicidal ideatio...,0,0,-0.009377
27,Automating all work should be seen as a humani...,0,1,0.001833
33,Shady Towing incident with City Towing Company,0,0,-0.010416
35,AITA for leaving my friends at a casino and ex...,1,1,0.064293


## Compare other text classifiers

Compare logistic regression, Complement Naive Bayes (suited to sparse word features), and SGD logistic classification against the SVM. All models use title + description, the same grouped training folds, and fold-specific TF-IDF fitting. Tune each family by toxic-class F1 and select the model family using cross-validation, not the test scores. One worker keeps resource use modest while labeling runs.

The same test set has already been inspected during development, so these test results are exploratory. Use a fresh untouched holdout for a final performance claim.


In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB

model_specs = {
    "Logistic regression": (
        LogisticRegression(solver="liblinear", max_iter=3000, random_state=42),
        {"classifier__C": [0.1, 1.0, 10.0], "classifier__class_weight": [None, "balanced"]},
    ),
    "Complement Naive Bayes": (
        ComplementNB(),
        {"classifier__alpha": [0.1, 0.5, 1.0, 2.0]},
    ),
    "SGD logistic": (
        SGDClassifier(loss="log_loss", max_iter=3000, tol=1e-4, random_state=42),
        {"classifier__alpha": [1e-5, 1e-4, 1e-3], "classifier__class_weight": [None, "balanced"]},
    ),
}
model_searches = {"Tuned SVM": search}
fitted_models = {"Tuned SVM": tuned_svm}
with TemporaryDirectory(prefix="reddit_model_comparison_") as cache_dir:
    memory = Memory(cache_dir, verbose=0)
    for name, (estimator, parameters) in model_specs.items():
        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(min_df=2, max_features=30000, sublinear_tf=True)),
            ("classifier", estimator),
        ], memory=memory)
        grid = {"tfidf__ngram_range": [(1, 1), (1, 2)], **parameters}
        model_search = GridSearchCV(
            pipeline, grid, cv=cv_splits,
            scoring={"toxic_f1": make_scorer(f1_score, zero_division=0),
                     "balanced_accuracy": "balanced_accuracy"},
            refit="toxic_f1", n_jobs=1, error_score="raise",
        )
        model_search.fit(X_train, y_train)
        model_searches[name] = model_search
        fitted_models[name] = model_search.best_estimator_.set_params(memory=None)
        print(f"{name}: CV F1 = {model_search.best_score_:.3f}; {model_search.best_params_}", flush=True)

cv_comparison = pd.DataFrame([
    {"model": name, "cv_toxic_f1": result.best_score_,
     "cv_f1_std": result.cv_results_["std_test_toxic_f1"][result.best_index_],
     "cv_balanced_accuracy": result.cv_results_["mean_test_balanced_accuracy"][result.best_index_]}
    for name, result in model_searches.items()
]).sort_values("cv_toxic_f1", ascending=False).reset_index(drop=True)
selected_model_name = cv_comparison.iloc[0]["model"]
selected_model = fitted_models[selected_model_name]
print("\nTraining cross-validation comparison:")
print(cv_comparison.to_string(index=False))
print("Selected using CV only:", selected_model_name)


In [ ]:
from sklearn.metrics import precision_score, recall_score

comparison_predictions = {
    "Majority baseline": baseline_pred,
    "Original SVM": original_pred,
    **{name: model.predict(X_test) for name, model in fitted_models.items()},
}
model_comparison = pd.DataFrame([
    {"model": name,
     "accuracy": accuracy_score(y_test, predictions),
     "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
     "toxic_precision": precision_score(y_test, predictions, zero_division=0),
     "toxic_recall": recall_score(y_test, predictions, zero_division=0),
     "toxic_f1": f1_score(y_test, predictions, zero_division=0)}
    for name, predictions in comparison_predictions.items()
])
print("Held-out comparison (exploratory; model selection uses CV above):")
print(model_comparison.to_string(index=False))
selected_predictions = comparison_predictions[selected_model_name]
print(f"\nCV-selected model: {selected_model_name}")
print(classification_report(y_test, selected_predictions, labels=[0, 1],
    target_names=TARGET_NAMES, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_test, selected_predictions, labels=[0, 1]),
    index=["actual 0", "actual 1"], columns=["predicted 0", "predicted 1"]))


## Try a neural network (multilayer perceptron)

Use the same title + description inputs and target. The pipeline converts text to TF-IDF, reduces the sparse features to 64 latent dimensions with truncated SVD, standardizes them, and trains a ReLU hidden layer followed by a binary output. SVD keeps the network small enough for this dataset. All feature transformations are fitted separately within each training fold.

Compare 32 versus 64 hidden neurons and two regularization strengths using the same three grouped folds. L-BFGS trains this small network without an additional internal validation split. Select settings by cross-validation toxic-class F1. This model uses scikit-learn and needs no PyTorch installation. Test scores remain exploratory because the holdout has been viewed repeatedly.


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

with TemporaryDirectory(prefix="reddit_mlp_cache_") as cache_dir:
    neural_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                                 max_features=30000, sublinear_tf=True)),
        ("svd", TruncatedSVD(n_components=64, random_state=42)),
        ("scale", StandardScaler()),
        ("network", MLPClassifier(activation="relu", solver="lbfgs",
                                  max_iter=3000, max_fun=50000, tol=1e-3, random_state=42)),
    ], memory=Memory(cache_dir, verbose=0))
    neural_search = GridSearchCV(
        neural_pipeline,
        {"network__hidden_layer_sizes": [(32,), (64,)],
         "network__alpha": [0.1, 1.0]},
        cv=cv_splits,
        scoring={"toxic_f1": make_scorer(f1_score, zero_division=0),
                 "balanced_accuracy": "balanced_accuracy"},
        refit="toxic_f1", n_jobs=1, error_score="raise", verbose=1,
    )
    # Limit numerical-library threads while the labeling job runs.
    with threadpool_limits(limits=1):
        neural_search.fit(X_train, y_train)
    neural_model = neural_search.best_estimator_.set_params(memory=None)

print("Neural-network settings:", neural_search.best_params_)
print(f"Neural-network CV toxic F1: {neural_search.best_score_:.3f}")
print("Training iterations:", neural_model.named_steps["network"].n_iter_)
print(f"SVD explained variance: {neural_model.named_steps['svd'].explained_variance_ratio_.sum():.1%}")
model_searches["Neural network"] = neural_search
fitted_models["Neural network"] = neural_model
# Choose the model family by training CV, not by test results.
selected_model_name = max(model_searches, key=lambda name: model_searches[name].best_score_)
selected_model = fitted_models[selected_model_name]
print("Selected model across all families by CV:", selected_model_name)


In [ ]:
neural_predictions = neural_model.predict(X_test)
neural_metrics = pd.DataFrame([{
    "model": "Neural network",
    "accuracy": accuracy_score(y_test, neural_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, neural_predictions),
    "toxic_precision": precision_score(y_test, neural_predictions, zero_division=0),
    "toxic_recall": recall_score(y_test, neural_predictions, zero_division=0),
    "toxic_f1": f1_score(y_test, neural_predictions, zero_division=0),
}])
# Replace the neural row on rerun instead of duplicating it.
model_comparison = pd.concat([
    model_comparison.loc[model_comparison["model"] != "Neural network"], neural_metrics
], ignore_index=True)
print(model_comparison.to_string(index=False))
print("\nNeural-network classification report:")
print(classification_report(y_test, neural_predictions, labels=[0, 1],
    target_names=TARGET_NAMES, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_test, neural_predictions, labels=[0, 1]),
    index=["actual 0", "actual 1"], columns=["predicted 0", "predicted 1"]))
